In [1]:
suppressPackageStartupMessages({
    library(ggplot2)
    library(ggpubr)
})

path <- "../2.current_version/"

In [12]:
species <- c("Hsap","Mmus","Pvit","Pmar")

get_OR <- function(species, type){
    df <- Reduce(rbind, lapply(species, FUN = function(s){
        wgd <- read.delim(paste0(path, s, "/", type ,"/", s, ".ohnolog_DEGs.fisher.celltype.txt"), header = T)
        ssd <- read.delim(paste0(path, s, "/", type ,"/", s, ".SSDparalog_DEGs.fisher.celltype.txt"), header = T)
        
        wgd$type <- "WGD"
        ssd$type <- "SSD"
    
        OR_info <- rbind(wgd, ssd)
        OR_info$species <- s
        return(OR_info)
    }))
    
    s = "Bflo"
    pa <- read.delim(paste0(path, s, "/", type ,"/", s, ".paralog_DEGs.fisher.celltype.txt"), header = T)
    pa$type <- "SSD"
    pa$species <- "Bflo"
    
    df <- rbind(df, pa)
    df$species <- factor(df$species, levels = c("Hsap","Mmus","Pvit","Pmar", "Bflo"))
    
    my_comparisons <- list(c("WGD", "SSD"))
    p <- ggboxplot(df, x = "type", y = "OR", color = "type", palette = "jco") + 
        stat_compare_means(comparisons = my_comparisons, method = "wilcox.test", paired = TRUE, label = "p.signif") + 
        stat_summary(fun = "median", geom = "text", aes(label = round(after_stat(y), 3)), vjust = -1) + 
        facet_wrap(~species, nrow = 1) +
        theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) + ylim(0, 4.5)
    ggsave(filename = paste0("OR.cell_family.summary.", type, ".pdf"), p, width = 5, height = 5.5)
}


In [13]:
get_OR(species, "wilcox")
get_OR(species, "roc")

Warning message:
“Removed 5 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 5 rows containing non-finite outside the scale range (`stat_signif()`).”
Warning message:
“Removed 5 rows containing non-finite outside the scale range
(`stat_summary()`).”
Warning message:
“Removed 12 rows containing missing values or values outside the scale range
(`geom_signif()`).”


In [4]:
species <- c("Hsap","Mmus","Pvit","Pmar")

get_ratio <- function(species, type){
    df <- Reduce(rbind, lapply(species, FUN = function(s){
        wgd <- read.delim(paste0(path, s, "/", type ,"/", s, ".ohnolog_ratio_inDEGs.stats.txt"), header = T)
        ssd <- read.delim(paste0(path, s, "/", type ,"/", s, ".SSDparalog_ratio_inDEGs.stats.txt"), header = T)
    
        wgd$type <- "WGD"
        ssd$type <- "SSD"
        colnames(wgd) <- colnames(ssd)
    
        info <- rbind(wgd, ssd)
        info$species <- s
        return(info)
    
    }))

    s = "Bflo"
    pa <- read.delim(paste0(path, s, "/", type ,"/", s, ".paralog_ratio_inDEGs.stats.txt"), header = T)
    pa$type <- "SSD"
    pa$species <- "Bflo"

    df <- rbind(df, pa)
    df$species <- factor(df$species, levels = c("Hsap","Mmus","Pvit","Pmar", "Bflo"))
    
    species <- c("Hsap","Mmus","Pvit","Pmar", "Bflo")

    ratio_bg <- Reduce(rbind, lapply(species, FUN = function(s){
        if (s == "Bflo"){
            tmp <- read.delim(paste0(path, s, "/", type ,"/", s, ".ratio_bg.txt"), header = F)
            tmp$V1 <- "SSD"
        } else {
            tmp <- read.delim(paste0(path, s, "/", type ,"/", s, ".ratio_bg.txt"), header = F)
            tmp <- tmp[tmp$V1 %in% c("ohnologs", "SSDparalogs"), ]
            tmp[tmp$V1 == "SSDparalogs", "V1"] <- "SSD"
            tmp[tmp$V1 == "ohnologs", "V1"] <- "WGD"
        }
        tmp$species <- s
        return(tmp)
    }))
    colnames(ratio_bg) <- c("type", "bg", "species")
    
    df <- merge(df, ratio_bg, by = c("species","type"))
    df$type <- factor(df$type, levels = c("WGD", "SSD"))
    
    p <- ggboxplot(df, x = "type", y = "paralogs.", color = "type", palette = "jco") + 
        stat_summary(fun = "median", geom = "text", aes(label = round(after_stat(y), 3)), vjust = -1) + 
        geom_hline(aes(yintercept = bg, color = type), linetype = "dashed") +
        facet_wrap(~species, nrow = 1) + 
        theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))
    ggsave(filename = paste0("paralog_ratio.cell_family.summary.", type, ".pdf"),p , width = 5, height = 5.5)
}


In [5]:
get_ratio(species, "wilcox")
get_ratio(species, "roc")

In [6]:
# plot family/number of paralogs ratio
species <- c("Hsap","Mmus","Pvit","Pmar")

get_fa_ratio <- function(species, type){
    df <- Reduce(rbind, lapply(species, FUN = function(s){
        wgd <- read.delim(paste0(path, s, "/", type ,"/", s, ".ohnolog_ratio_inDEGs.stats.txt"), header = T)
        ssd <- read.delim(paste0(path, s, "/", type ,"/", s, ".SSDparalog_ratio_inDEGs.stats.txt"), header = T)
    
        wgd$type <- "WGD"
        ssd$type <- "SSD"
        colnames(wgd) <- colnames(ssd)
    
        info <- rbind(wgd, ssd)
        info$species <- s
        return(info)
    
    }))

    s = "Bflo"
    pa <- read.delim(paste0(path, s, "/", type ,"/", s, ".paralog_ratio_inDEGs.stats.txt"), header = T)
    pa$type <- "SSD"
    pa$species <- "Bflo"

    df <- rbind(df, pa)
    df$species <- factor(df$species, levels = c("Hsap","Mmus","Pvit","Pmar", "Bflo"))
    
    species <- c("Hsap","Mmus","Pvit","Pmar", "Bflo")

    ratio_bg <- Reduce(rbind, lapply(species, FUN = function(s){
        if (s == "Bflo"){
            tmp <- read.delim(paste0(path, s, "/", type ,"/", s, ".family_ratio_bg.txt"), header = F)
            tmp$V1 <- "SSD"
        } else {
            tmp <- read.delim(paste0(path, s, "/", type ,"/", s, ".family_ratio_bg.txt"), header = F)
            tmp <- tmp[tmp$V1 %in% c("ohnologs", "SSDparalogs"), ]
            tmp[tmp$V1 == "SSDparalogs", "V1"] <- "SSD"
            tmp[tmp$V1 == "ohnologs", "V1"] <- "WGD"
        }
        tmp$species <- s
        return(tmp)
    }))
    colnames(ratio_bg) <- c("type", "bg", "species")
    
    df <- merge(df, ratio_bg, by = c("species","type"))
    df$type <- factor(df$type, levels = c("WGD", "SSD"))
    
    p <- ggboxplot(df, x = "type", y = "families_divided_by_paralogs.", color = "type", palette = "jco") + 
        stat_summary(fun = "median", geom = "text", aes(label = round(after_stat(y), 3)), vjust = -1) + 
        geom_hline(aes(yintercept = bg, color = type), linetype = "dashed") +
        facet_wrap(~species, nrow = 1) + 
        theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))
    ggsave(filename = paste0("paralog_family_ratio.switching_within_species.cell_family.summary.", type, ".pdf"),p , width = 5, height = 5.5)
}

In [7]:
get_fa_ratio(species, "wilcox")
get_fa_ratio(species, "roc")

Warning message:
“Removed 1 row containing non-finite outside the scale range (`stat_boxplot()`).”
Warning message:
“Removed 1 row containing non-finite outside the scale range (`stat_summary()`).”


In [8]:
# one sample t-test p-values were computed below and added to plots


In [22]:
df <- get_ratio(species, "wilcox")

In [24]:
unique(df[,c('species', 'type', 'bg')])

,species,type,bg
,<fct>,<fct>,<dbl>
1,Bflo,SSD,0.42
16,Hsap,SSD,0.28
33,Hsap,WGD,0.31
50,Mmus,SSD,0.25
72,Mmus,WGD,0.32
94,Pmar,SSD,0.27
112,Pmar,WGD,0.23
130,Pvit,SSD,0.27
147,Pvit,WGD,0.30


In [25]:
wilcox.test(df[df$type == 'SSD' & df$species == 'Bflo', 'paralogs.'], mu = 0.42, alternative = "two.sided")
wilcox.test(df[df$type == 'SSD' & df$species == 'Pmar', 'paralogs.'], mu = 0.27, alternative = "two.sided")
wilcox.test(df[df$type == 'SSD' & df$species == 'Pvit', 'paralogs.'], mu = 0.27, alternative = "two.sided")
wilcox.test(df[df$type == 'SSD' & df$species == 'Mmus', 'paralogs.'], mu = 0.25, alternative = "two.sided")
wilcox.test(df[df$type == 'SSD' & df$species == 'Hsap', 'paralogs.'], mu = 0.28, alternative = "two.sided")


	Wilcoxon signed rank exact test

data:  df[df$type == "SSD" & df$species == "Bflo", "paralogs."]
V = 10, p-value = 0.002625
alternative hypothesis: true location is not equal to 0.42


Warning message in wilcox.test.default(df[df$type == "SSD" & df$species == "Pmar", :
“cannot compute exact p-value with ties”



	Wilcoxon signed rank test with continuity correction

data:  df[df$type == "SSD" & df$species == "Pmar", "paralogs."]
V = 0, p-value = 0.0002137
alternative hypothesis: true location is not equal to 0.27



	Wilcoxon signed rank exact test

data:  df[df$type == "SSD" & df$species == "Pvit", "paralogs."]
V = 42, p-value = 0.1089
alternative hypothesis: true location is not equal to 0.27



	Wilcoxon signed rank exact test

data:  df[df$type == "SSD" & df$species == "Mmus", "paralogs."]
V = 46, p-value = 0.007443
alternative hypothesis: true location is not equal to 0.25



	Wilcoxon signed rank exact test

data:  df[df$type == "SSD" & df$species == "Hsap", "paralogs."]
V = 2, p-value = 4.578e-05
alternative hypothesis: true location is not equal to 0.28


In [26]:
wilcox.test(df[df$type == 'WGD' & df$species == 'Pmar', 'paralogs.'], mu = 0.23, alternative = "two.sided")
wilcox.test(df[df$type == 'WGD' & df$species == 'Pvit', 'paralogs.'], mu = 0.30, alternative = "two.sided")
wilcox.test(df[df$type == 'WGD' & df$species == 'Mmus', 'paralogs.'], mu = 0.32, alternative = "two.sided")
wilcox.test(df[df$type == 'WGD' & df$species == 'Hsap', 'paralogs.'], mu = 0.31, alternative = "two.sided")


	Wilcoxon signed rank exact test

data:  df[df$type == "WGD" & df$species == "Pmar", "paralogs."]
V = 171, p-value = 7.629e-06
alternative hypothesis: true location is not equal to 0.23



	Wilcoxon signed rank exact test

data:  df[df$type == "WGD" & df$species == "Pvit", "paralogs."]
V = 143, p-value = 0.0006561
alternative hypothesis: true location is not equal to 0.3



	Wilcoxon signed rank exact test

data:  df[df$type == "WGD" & df$species == "Mmus", "paralogs."]
V = 250, p-value = 2.384e-06
alternative hypothesis: true location is not equal to 0.32



	Wilcoxon signed rank exact test

data:  df[df$type == "WGD" & df$species == "Hsap", "paralogs."]
V = 151, p-value = 4.578e-05
alternative hypothesis: true location is not equal to 0.31
